In [1]:
import os
import librosa
import numpy as np
from transformers import BertTokenizer, BertModel
import torch

/Users/mohamedaminemrabet/miniforge3/envs/tfnew/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('../data/musiccaps-public.csv')

In [ ]:
df.head()

,ytid,start_s,end_s,audioset_positive_labels,aspect_list,caption,author_id,is_balanced_subset,is_audioset_eval
0,-0Gj8-vB1q4,30,40,"/m/0140xf,/m/02cjck,/m/04rlf","['low quality', 'sustained strings melody', 's...",The low quality recording features a ballad so...,4,False,True
1,-0SdAVK79lg,30,40,"/m/0155w,/m/01lyv,/m/0342h,/m/042v_gx,/m/04rlf...","['guitar song', 'piano backing', 'simple percu...",This song features an electric guitar as the m...,0,False,False
2,-0vPFx-wRRI,30,40,"/m/025_jnm,/m/04rlf","['amateur recording', 'finger snipping', 'male...",a male voice is singing a melody with changing...,6,False,True
3,-0xzrMun0Rs,30,40,"/m/01g90h,/m/04rlf","['backing track', 'jazzy', 'digital drums', 'p...",This song contains digital drums playing a sim...,6,False,True
4,-1LrH01Ei1w,30,40,"/m/02p0sh1,/m/04rlf","['rubab instrument', 'repetitive melody on dif...",This song features a rubber instrument being p...,0,False,False


In [27]:
exist = []
for file in os.listdir('../data/music_data'):
    if file[:-4] in list(df['ytid']):
        exist.append(file[:-4])

In [29]:
filtered_df = df[df['ytid'].isin(exist)]

In [30]:
filtered_df

,ytid,start_s,end_s,audioset_positive_labels,aspect_list,caption,author_id,is_balanced_subset,is_audioset_eval
0,-0Gj8-vB1q4,30,40,"/m/0140xf,/m/02cjck,/m/04rlf","['low quality', 'sustained strings melody', 's...",The low quality recording features a ballad so...,4,False,True
1,-0SdAVK79lg,30,40,"/m/0155w,/m/01lyv,/m/0342h,/m/042v_gx,/m/04rlf...","['guitar song', 'piano backing', 'simple percu...",This song features an electric guitar as the m...,0,False,False
2,-0vPFx-wRRI,30,40,"/m/025_jnm,/m/04rlf","['amateur recording', 'finger snipping', 'male...",a male voice is singing a melody with changing...,6,False,True
3,-0xzrMun0Rs,30,40,"/m/01g90h,/m/04rlf","['backing track', 'jazzy', 'digital drums', 'p...",This song contains digital drums playing a sim...,6,False,True
4,-1LrH01Ei1w,30,40,"/m/02p0sh1,/m/04rlf","['rubab instrument', 'repetitive melody on dif...",This song features a rubber instrument being p...,0,False,False
...,...,...,...,...,...,...,...,...,...
995,9Kut4r8hswE,30,40,"/m/04rlf,/m/07gxw,/m/07lnk,/m/0m0jc","['low quality', 'electro', 'noisy traffic soun...",The low quality recording features an electro ...,4,False,True
996,9L6ePkWtZI4,30,40,"/m/04rlf,/m/0gg8l","['low quality', 'acoustic sitar chord progress...",The low quality recording features a cover of ...,4,False,True
997,9Lst8RagMYs,500,510,"/m/04rlf,/m/06rqw","['ska song', 'no voices', 'instrumental', 'bra...",This ska song features the main melody played ...,0,False,False
998,9M4IT3lOU10,30,40,"/m/04rlf,/m/07swgks","['low quality music', 'latin rhythm', 'male vo...",This clip features very low quality recorded m...,0,False,True


In [ ]:
# Set paths
DATASET_PATH = "../data/music_data/"
AUDIO_PATH = os.path.join(DATASET_PATH, "audio_files")
TEXT_FILE = os.path.join(DATASET_PATH, "text_descriptions.csv")

In [4]:
# 1. Text Preprocessing
def preprocess_text(text_descriptions):
    """
    Tokenize and encode text descriptions using BERT.
    """
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    model = BertModel.from_pretrained("bert-base-uncased")
    embeddings = []

    for text in text_descriptions:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=128)
        with torch.no_grad():
            outputs = model(**inputs)
        # Use the [CLS] token embedding
        cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze(0).numpy()
        embeddings.append(cls_embedding)
    
    return np.array(embeddings)

In [8]:
# 2. Audio Preprocessing
def preprocess_audio(audio_path, target_sr=16000, n_mels=128):
    """
    Convert audio files to mel spectrograms.
    """
    audio_files = [f for f in os.listdir(audio_path) if f.endswith(".wav")]
    spectrograms = []

    for file in audio_files:
        file_path = os.path.join(audio_path, file)
        try:
            # Load audio
            y, sr = librosa.load(file_path, sr=target_sr)
            # Convert to mel spectrogram
            mel_spec = librosa.feature.melspectrogram(y, sr=sr, n_mels=n_mels)
            # Convert to log scale
            log_mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
            spectrograms.append(log_mel_spec)
        except Exception as e:
            print(f"Error processing {file}: {e}")
    
    return np.array(spectrograms)

In [7]:
# 3. Main Function
def preprocess_data(text_file, audio_path):
    """
    Preprocess text and audio data.
    """
    # Load text descriptions
    import pandas as pd
    text_df = pd.read_csv(text_file)
    text_descriptions = text_df['text'].tolist()

    # Preprocess text and audio
    text_embeddings = preprocess_text(text_descriptions)
    audio_features = preprocess_audio(audio_path)

    return text_embeddings, audio_features

In [ ]:
# Preprocess the data
text_embeddings, audio_features = preprocess_data(TEXT_FILE, AUDIO_PATH)

# Save preprocessed data
np.save("text_embeddings.npy", text_embeddings)
np.save("audio_features.npy", audio_features)